## Objective

The objective of this analysis is to develop and compare machine-learning models for predicting stroke occurrence.

Three models are evaluated:

- **Logistic Regression** — an interpretable linear baseline.
- **Random Forest** — a tree-based ensemble that can capture nonlinear relationships and interactions.
- **XGBoost** — a gradient-boosted tree model designed to capture more complex predictive patterns.

The analysis focuses on four aspects of model performance:

1. **Discrimination** — how well the model separates individuals with and without stroke.
2. **Calibration** — how closely predicted probabilities correspond to observed stroke frequency.
3. **Risk stratification** — whether predicted probabilities can separate individuals into groups with different observed stroke rates.

The final model is subsequently examined for feature importance and interpretability.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier   
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    classification_report,
    confusion_matrix
)

from sklearn.calibration import calibration_curve
import warnings

In [2]:
warnings.filterwarnings("ignore")

## Data Loading

The dataset has already been divided into **training** and **held-out test** sets during the preceding data-preparation/EDA workflow.

- The **training set** is used for preprocessing, model fitting, and hyperparameter tuning.
- The **test set** is kept separate and used to assess performance on previously unseen observations.

The identifier (`id`) is excluded from the predictors, while `stroke` is used as the binary outcome.

In [3]:
train_df = pd.read_parquet('../data/processed/train_df.parquet')
train_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke,bmi_missing
0,25283,Female,48.0,0,0,Yes,Private,Urban,69.21,33.1,Never smoked,0,0
1,41917,Female,29.0,0,0,No,Private,Urban,84.19,21.2,Never smoked,0,0
2,36045,Female,35.0,0,0,Yes,Private,Rural,119.40,22.9,Never smoked,0,0
3,28315,Male,38.0,0,0,Yes,Private,Rural,108.68,32.7,Never smoked,0,0
4,26325,Male,14.0,0,0,No,Government Job,Urban,82.34,31.6,Unknown,0,0


In [4]:
test_df = pd.read_parquet('../data/processed/test_df.parquet')
test_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke,bmi_missing
0,3777,Female,28.0,1,0,Yes,Government Job,Rural,83.66,36.4,Never smoked,0,0
1,8277,Female,3.0,0,0,No,Not working,Urban,93.30,19.5,Unknown,0,0
2,58282,Female,53.0,0,0,Yes,Government Job,Rural,64.40,31.0,Smokes,0,0
3,46488,Male,35.0,0,0,Yes,Private,Rural,69.22,42.8,Never smoked,0,0
4,8831,Female,58.0,0,0,Yes,Private,Rural,94.30,29.1,Unknown,0,0


In [5]:
X_train = train_df.drop(columns = ['id','stroke'])
y_train = train_df['stroke']

X_test = test_df.drop(columns = ['id','stroke'])
y_test = test_df['stroke']

In [6]:
print("Training stroke prevalence:", y_train.mean()) 
print("Test stroke prevalence:", y_test.mean())

Training stroke prevalence: 0.04869097137264497
Test stroke prevalence: 0.04892367906066536


## Preprocessing

### Numerical variables
Numerical predictors are standardized for Logistic Regression.

### Categorical variables
Categorical predictors are converted to indicator variables using one-hot encoding. The first category is dropped to provide a reference category and avoid redundant columns.

### Missing BMI values
BMI missingness was addressed during the preceding EDA workflow using an age- and gender-informed imputation strategy. A separate `bmi_missing` indicator is retained so that the model can distinguish observed BMI values from values that were originally missing.

In [7]:
cat_vars = ['gender','ever_married', 'work_type', 'residence_type','smoking_status']
num_vars = ['age', 'avg_glucose_level', 'bmi', 'bmi_missing', 'hypertension', 'heart_disease']

In [8]:
# Logistic regression preprocessing
num_pipeline_lr = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop = 'first'))
])

preprocessor_lr = ColumnTransformer([
    ('num', num_pipeline_lr, num_vars),
    ('cat', cat_pipeline, cat_vars)
])

# Tree-model preprocessing
preprocessor_tree = ColumnTransformer([
    ('num', 'passthrough', num_vars),
    ('cat', cat_pipeline, cat_vars)
])

## Model Training and Hyperparameter Tuning

The three candidate models are trained using the same training dataset and evaluated using **5-fold stratified cross-validation**.

### Why PR-AUC?

Stroke is an uncommon outcome in this dataset, with a prevalence of approximately **4.9%**. With an imbalanced outcome, accuracy can be misleading because a model could achieve high accuracy simply by predicting the majority class.

In [9]:
# Logistic Regression Model
lr = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# Random Forest
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# XGBoost
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

In [10]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_lr),
    ('model', lr)
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', rf)
])

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', xgb)
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metric = 'average_precision'

In [12]:
# LR Hyperparameter Tuning
lr_param_grid = {
    'model__C': [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=cv,
    scoring=scoring_metric,
    n_jobs=-1,
    refit=True
)

lr_grid.fit(X_train, y_train)
print("Best LR parameters:")
print(lr_grid.best_params_)

print("\nBest LR CV PR-AUC:")
print(lr_grid.best_score_)

best_lr = lr_grid.best_estimator_

Best LR parameters:
{'model__C': 1}

Best LR CV PR-AUC:
0.22983681087607236


In [13]:
# RF Hyperparameter Tuning
rf_param_grid = {
    'model__n_estimators': [
        300, 500
    ],

    'model__max_depth': [
        None,
        5, 10
    ],

    'model__min_samples_leaf': [
        1, 5, 10
    ],

    'model__max_features': [
        'sqrt', 'log2'
    ]
}
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring=scoring_metric,
    n_jobs=-1,
    refit=True
)

rf_grid.fit(X_train, y_train)
print("Best RF parameters:")
print(rf_grid.best_params_)

print("\nBest RF CV PR-AUC:")
print(rf_grid.best_score_)

best_rf = rf_grid.best_estimator_

Best RF parameters:
{'model__max_depth': 5, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 300}

Best RF CV PR-AUC:
0.2693102530442009


In [14]:
# XGB Hyperparameter Tuning
xgb_param_dist = {
    'model__n_estimators': [
        200,
        300,
        400,
        500
    ],

    'model__max_depth': [
        2,
        3,
        4
    ],

    'model__learning_rate': [
        0.03,
        0.05,
        0.1
    ],

    'model__subsample': [
        0.8,
        1.0
    ],

    'model__colsample_bytree': [
        0.8,
        1.0
    ],

    'model__min_child_weight': [
        1,
        3,
        5
    ]
}
xgb_random = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=30,
    cv=cv,
    scoring=scoring_metric,
    n_jobs=-1,
    random_state=42,
    refit=True
)

xgb_random.fit(X_train, y_train)
print("Best XGBoost parameters:")
print(xgb_random.best_params_)

print("\nBest XGBoost CV PR-AUC:")
print(xgb_random.best_score_)
best_xgb = xgb_random.best_estimator_

Best XGBoost parameters:
{'model__subsample': 0.8, 'model__n_estimators': 200, 'model__min_child_weight': 5, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}

Best XGBoost CV PR-AUC:
0.2285231013154138


## Cross-Validation Model Comparison

The best hyperparameter configuration for each model is identified using 5-fold stratified cross-validation.

This section compares the resulting **best CV PR-AUC** values across Logistic Regression, Random Forest, and XGBoost.

These values are used to understand how each model performed during cross-validation. The **Random Forest** model scored the highest CV PR-AUC **(0.269)**


In [15]:
cv_comparison = pd.DataFrame({
    'Model' : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Best CV PR-AUC' : [lr_grid.best_score_, rf_grid.best_score_, xgb_random.best_score_]
})

cv_comparison.sort_values('Best CV PR-AUC', ascending=False)

,Model,Best CV PR-AUC
1,Random Forest,0.269310
0,Logistic Regression,0.229837
2,XGBoost,0.228523


### Evaluating model on Test set
The models performed well on unseen data across all three metrics of ROC-AUC, PR-AUC and Brier score, indicating good generalization.

In [16]:
models = {
    'Logistic Regression': best_lr,
    'Random Forest': best_rf,
    'XGBoost': best_xgb
}

# Generating probability predictions:

test_probabilities = {}

for name, model in models.items():

    test_probabilities[name] = model.predict_proba(X_test)[:, 1]

In [17]:
results = []

for name, y_proba in test_probabilities.items():

    roc_auc = roc_auc_score(
        y_test,
        y_proba
    )

    pr_auc = average_precision_score(
        y_test,
        y_proba
    )

    brier = brier_score_loss(
        y_test,
        y_proba
    )

    results.append({
        'Model': name,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc,
        'Brier Score': brier
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    'PR-AUC',
    ascending=False
)

,Model,ROC-AUC,PR-AUC,Brier Score
2,XGBoost,0.826605,0.239840,0.041814
0,Logistic Regression,0.836790,0.218966,0.041849
1,Random Forest,0.831193,0.209818,0.042267
